In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import scipy.io as sio
import numpy as np
import os
import time
import wandb
from tqdm import tqdm
import copy
import matplotlib.pyplot as plt

np.random.seed(2137)
torch.manual_seed(2137)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

# Tutaj zaczyna sie jescze zad0, dopiero pomte bedzie zad1

In [ ]:
labels_path = "./data/imagelabels.mat"
setid_path = "./data/setid.mat"
img_dir = "./data/jpg"

# słowniki ze zmiennymi z matlaba

labels_data = sio.loadmat(labels_path) 
setid_data = sio.loadmat(setid_path)

In [ ]:
print(f"labels dict: {labels_data.keys()}")

print(f"setid dict: {setid_data.keys()}")

W labelsach jest są labelsy od 1 do 102, to bedzimy chcieli zamienic na 0 do 101, bo do loss bedzie wygodniejsze.   
A w dancyh trenigowych są też od 1 do 8189, tutaj nie ma to wiekszego znaczenia bo to jedynie ids do zdjeć. 

In [ ]:
labels = labels_data['labels'].squeeze() - 1

train_ids = setid_data['trnid'].squeeze()
val_ids = setid_data['valid'].squeeze()
test_ids = setid_data['tstid'].squeeze()

print(f"Liczba zdjęć: Treningowe ({len(train_ids)}), Walidacyjne ({len(val_ids)}), Testowe ({len(test_ids)})")

In [ ]:
class FlowersDataset(Dataset):
    def __init__(self, ids, labels, img_dir, transform=None):
        self.ids = ids
        self.labels = labels
        self.img_dir = img_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.ids)
    
    def __getitem__(self, index):
        img_id = self.ids[index]

        img_name = f"image_{img_id:05d}.jpg"
        img_path = os.path.join(self.img_dir, img_name)

        img = Image.open(img_path).convert("RGB")

        label = self.labels[img_id - 1]

        if self.transform:
            img = self.transform(img)
        
        return img, label

In [ ]:
basic_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [ ]:
train_dataset = FlowersDataset(train_ids, labels, img_dir, basic_transform)
val_dataset = FlowersDataset(val_ids, labels, img_dir, basic_transform)
test_dataset = FlowersDataset(test_ids, labels, img_dir, basic_transform)

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
print("\nPobieram próbny batch z DataLoadera...")
images, targets = next(iter(train_loader))
print(f"Kształt paczki obrazków: {images.shape} -> (Batch, Kanały, Wysokość, Szerokość)")
print(f"Kształt paczki etykiet: {targets.shape}")

In [ ]:
class ExampleCNN(nn.Module):
    def __init__(self, num_classes=102):
        super(ExampleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        # Fully connected layers
        self.c = 56*56*16
        self.fc1 = nn.Linear(self.c, 1024) # Flatten (512 * 7 * 7) features
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(1024, num_classes)
    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.view(-1, self.c) # Flatten the tensor
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x) # logits for all classes
        return x

In [ ]:
model_cnn = ExampleCNN(num_classes=102).to(device)
print(model_cnn)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_cnn.parameters(), lr=0.001)

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, lr, name, architecture):
    wandb.init(
        project="flowers",
        name=name,      
        config={                      
            "learning_rate": lr,
            "architecture": architecture,
            "epochs": epochs,
            "batch_size": batch_size
        }
    )
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    start_time = time.time()

    for epoch in range(epochs):
        print(f"Epoka {epoch+1}/{epochs}")

        model.train()
        running_loss = 0.0
        running_correct = 0

        for inputs, labels in tqdm(train_loader, desc="Traning"):
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            running_correct += torch.sum(preds == labels.data)

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = running_correct / len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        val_correct = 0

        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc="Validation"):
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                _, preds = torch.max(outputs, 1)
                val_loss += loss.item() * inputs.size(0)
                val_correct += torch.sum(preds == labels.data)
        
        epoch_val_loss = val_loss / len(val_loader.dataset)
        epoch_val_acc = val_correct / len(val_loader.dataset)

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": epoch_train_loss,
            "train_acc": epoch_train_acc,
            "val_loss": epoch_val_loss,
            "val_acc": epoch_val_acc
        })

        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)
        
        print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
        print(f"Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.4f}\n")

    total_time = time.time() - start_time
    print(f"Trening zakończony w {total_time // 60:.0f}m {total_time % 60:.0f}s")
        
    wandb.log({"total_time_seconds": total_time})
    
    wandb.finish()
    
    return history

In [ ]:
# history_cnn = train_model(model_cnn, train_loader, val_loader, criterion, optimizer, epochs=10, lr=0.01, name="task0-cnn", architecture="example_cnn")

## Treninig 10%

In [ ]:
num_10_percent = len(train_ids) // 10
train_ids_10 = train_ids[:num_10_percent]

print(f"Oryginalny zbiór treningowy: {len(train_ids)} zdjęć.")
print(f"Zbiór 10%: {len(train_ids_10)} zdjęć.")

train_dataset_10 = FlowersDataset(train_ids_10, labels, img_dir, transform=basic_transform)
train_loader_10 = DataLoader(train_dataset_10, batch_size=batch_size, shuffle=True)

model_cnn_10 = ExampleCNN(num_classes=102).to(device)

optimizer_10 = optim.Adam(model_cnn_10.parameters(), lr=0.001)

print("\nRozpoczynamy trening na 10% danych...")
# history_cnn_10 = train_model(
#     model=model_cnn_10, 
#     train_loader=train_loader_10, 
#     val_loader=val_loader, 
#     criterion=criterion, 
#     optimizer=optimizer_10, 
#     epochs=10, 
#     lr=0.001, 
#     name="task0-cnn-10-percent", 
#     architecture="example_cnn"
# )

# VGG13

## for scratch

W pracy tutaj była odpowiedź na pytania z zadania:

On a system equipped with
four NVIDIA Titan Black GPUs, training a single net took 2–3 weeks depending on the architecture.

ILSVRC-2012 dataset (which was used for ILSVRC 2012–2014 challenges). The dataset includes images of 1000 classes, and is split into three sets: training (1.3M
images), validation (50K images), and testing (100K images with held-out class labels). 

In [ ]:
vgg13_from_scratch = models.vgg13(weights=None, num_classes=102).to(device)

total_params_vgg = sum(p.numel() for p in vgg13_from_scratch.parameters())
print(f"Liczba parametrów w VGG-13: {total_params_vgg:,}")

optimizer_vgg = optim.Adam(vgg13_from_scratch.parameters(), lr=0.01)

print("\nRozpoczynamy próbę trenowania VGG-13 od zera...")

# history_vgg_scratch = train_model(
#     model=vgg13_from_scratch, 
#     train_loader=train_loader, 
#     val_loader=val_loader, 
#     criterion=criterion, 
#     optimizer=optimizer_vgg, 
#     epochs=10, 
#     lr=0.01, 
#     name="task0-vgg13-scratch", 
#     architecture="vgg13_scratch"
# )

Trenowanie głębokich sieci (jak VGG) od zera na małym zbiorze danych jest nieefektywne, zajmuje mnóstwo czasu i prowadzi do dobrych wyników. Sieć nie potrafi nauczyć się tak skomplikowanych cech z tak małej próbki. Dlatego Transfer Learning może być koniecznością.

# Zad1

In [ ]:
vgg13_weights = models.VGG13_Weights.DEFAULT
vgg_transforms = vgg13_weights.transforms() 

train_dataset_tl = FlowersDataset(train_ids, labels, img_dir, transform=vgg_transforms)
val_dataset_tl = FlowersDataset(val_ids, labels, img_dir, transform=vgg_transforms)
test_dataset_tl = FlowersDataset(test_ids, labels, img_dir, transform=vgg_transforms)

train_loader_tl = DataLoader(train_dataset_tl, batch_size=batch_size, shuffle=True)
val_loader_tl = DataLoader(val_dataset_tl, batch_size=batch_size, shuffle=False)
test_loader_tl = DataLoader(test_dataset_tl, batch_size=batch_size, shuffle=False)

vgg13_tl = models.vgg13(weights=vgg13_weights)

In [ ]:
# zamrażamy wagi

for param in vgg13_tl.features.parameters():
    param.requires_grad = False

for param in vgg13_tl.classifier.parameters():
    param.requires_grad = False

# PODMIENIAMY OSTATNIĄ WARSTWĘ 
in_features = vgg13_tl.classifier[6].in_features
# Ta nowa warstwa domyślnie MA requires_grad = True
vgg13_tl.classifier[6] = nn.Linear(in_features, 102)

vgg13_tl = vgg13_tl.to(device)

params_to_update = filter(lambda p: p.requires_grad, vgg13_tl.parameters())
optimizer_tl = optim.Adam(params_to_update, lr=0.01)

print("Rozpoczynamy potężny Transfer Learning! (Uczymy TYLKO ostatnią warstwę)")
history_tl = train_model(
    model=vgg13_tl,
    train_loader=train_loader_tl,
    val_loader=val_loader_tl,
    criterion=criterion,
    optimizer=optimizer_tl,
    epochs=10, 
    lr=0.01,
    name="task1.1-transfer-learning",
    architecture="vgg13_frozen"
)

In [ ]:
def evaluate_on_test(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Testing"):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    test_acc = correct / total
    print(f"Acc: {test_acc * 100:.2f}%")
    return test_acc

torch.save(vgg13_tl.state_dict(), "vgg13_task1.1_last_layer_only.pth")

evaluate_on_test(vgg13_tl, test_loader_tl)

In [ ]:
def run_prefix_experiment(k_frozen, train_loader, val_loader, device):
    print(f"\n{'='*50}")
    print(f"Zamrażanie pierwszych {k_frozen} warstw Conv2d")
    print(f"{'='*50}")
    
    wandb.init(
        project="flowers",
        name=f"task1.2-freeze-{k_frozen}",
        config={"k_frozen": k_frozen, "batch_size": batch_size, "learning_rate": 0.001}
    )
    
    model = models.vgg13(weights=models.VGG13_Weights.DEFAULT)
    in_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(in_features, 102)
    
    conv_layers = [m for m in model.features.modules() if isinstance(m, nn.Conv2d)]
    
    for i, conv in enumerate(conv_layers):
        if i < k_frozen:
            for param in conv.parameters():
                param.requires_grad = False  # ZAMROŻONE 
        else:
            for param in conv.parameters():
                param.requires_grad = True   # ODMROŻONE
                
    for param in model.classifier.parameters():
        param.requires_grad = True
        
    model = model.to(device)
    
    params_to_update = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = optim.Adam(params_to_update, lr=0.001) 
    criterion = nn.CrossEntropyLoss()
    
    history = {'val_acc': [], 'epoch_times': [], 'epochs_to_95': None, 'time_to_95': None}
    best_acc = 0.0
    best_weights = copy.deepcopy(model.state_dict())
    
    patience = 4 # Czekamy 4 epoki na poprawę, potem kończymy ("convergence")
    epochs_no_improve = 0
    max_epochs = 20 
    
    start_time = time.time()
    
    for epoch in range(max_epochs):
        epoch_start = time.time()
        
        model.train()
        for inputs, labels in tqdm(train_loader, desc=f"Epoka {epoch+1} (Trening)", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f"Epoka {epoch+1} (Walidacja)", leave=False):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_correct += torch.sum(preds == labels.data)
                
        val_acc = (val_correct.float() / len(val_loader.dataset)).item()
        epoch_time = time.time() - epoch_start
        
        history['val_acc'].append(val_acc)
        history['epoch_times'].append(epoch_time)
        
        print(f"  Epoka {epoch+1} | Czas: {epoch_time:.1f}s | Val Acc: {val_acc:.4f}")
        wandb.log({"epoch": epoch+1, "val_acc": val_acc, "epoch_time": epoch_time})
        
        if val_acc >= 0.95 and history['epochs_to_95'] is None:
            history['epochs_to_95'] = epoch + 1
            history['time_to_95'] = time.time() - start_time
            print(f"OSIĄGNIĘTO 95% W EPOCE {epoch + 1}!")
            
        if val_acc > best_acc:
            best_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= patience:
            print(f"Brak poprawy od {patience} epok.")
            break
            
    total_time = time.time() - start_time
    history['total_time'] = total_time
    
    model.load_state_dict(best_weights)
    save_path = f"vgg13_frozen_{k_frozen}_layers.pth"
    torch.save(model.state_dict(), save_path)
    print(f"Zapisano {save_path} (Najlepsze Acc: {best_acc:.4f})")
    
    wandb.finish()
    return history

In [ ]:
all_experiment_results = {}

In [ ]:
for k in range(1, 7):
    results = run_prefix_experiment(
        k_frozen=k, 
        train_loader=train_loader_tl, 
        val_loader=val_loader_tl, 
        device=device
    )
    all_experiment_results[k] = results

# Strasznie długo mi się to uczy ...

# Trzeba dodać wizualizacje wyników